# Instruction-Tuning with Large Language Models (LLMs)

Instruction-based fine-tuning, commonly known as *Instruction GPT*, is a training approach where language models are optimized to understand and follow human instructions accurately. The objective is to help the model generate meaningful and task-specific responses based on user prompts.

In instruction-tuning, the dataset is highly important because it provides structured examples containing instructions, optional context, and expected outputs. By learning from these examples, the model becomes capable of handling a wide range of real-world tasks more effectively.

Many Instruction GPT systems are further improved using human feedback to refine response quality and alignment. However, this project focuses only on the instruction fine-tuning process and does not include reinforcement learning or human feedback optimization.

The instruction and context are typically merged into a single input sequence so the model can better understand the task and generate an appropriate response.

## Key Components

- **Instruction**  
  A task description or command that tells the model what action to perform.

- **Context**  
  Additional background information or supporting details required to complete the task correctly.

- **Combined Input**  
  The instruction and context are combined into one sequence before being passed to the model during training or inference.


### **Import Libraries**

In [ ]:
import torch
import torch.nn as nn
from tqdm import tqdm
import matplotlib.pyplot as plt
from datasets import load_dataset 
from torch.utils.data.dataset import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Pipeline

### **Load Dataset**

In [2]:
SEED = 42

In [3]:
dataset = load_dataset("sahil2801/CodeAlpaca-20k")

In [4]:
dataset.column_names

{'train': ['output', 'instruction', 'input']}

In [5]:
dataset['train'][1000]

{'output': 's = "Hello world" \ns = s[::-1] \nprint(s)',
 'instruction': 'Reverse the string given in the input',
 'input': 'Hello world'}

In [6]:
dataset = dataset['train']
dataset[1000]

{'output': 's = "Hello world" \ns = s[::-1] \nprint(s)',
 'instruction': 'Reverse the string given in the input',
 'input': 'Hello world'}

**Filter the Dataset Which Don't Have Any Input**

In [7]:
dataset = dataset.filter(lambda example: example['input'] == '')
dataset[0:2]

{'output': ['arr = [2, 4, 6, 8, 10]',
  'Height of triangle = opposite side length * sin (angle) / side length'],
 'instruction': ['Create an array of length 5 which contains all even numbers between 1 and 10.',
  'Formulate an equation to calculate the height of a triangle given the angle, side lengths and opposite side length.'],
 'input': ['', '']}

In [8]:
### shuffle the dataset
dataset = dataset.shuffle(seed=SEED)

In [9]:
dataset

Dataset({
    features: ['output', 'instruction', 'input'],
    num_rows: 9764
})

In [10]:
dataset['instruction'][0]

'Create a function that produces input strings for a calculator.'

**Split Dataset**

In [11]:
train_test_split = dataset.train_test_split(test_size=0.2,seed=SEED)
dataset_train = train_test_split['train']
dataset_test = train_test_split['test']
train_test_split

DatasetDict({
    train: Dataset({
        features: ['output', 'instruction', 'input'],
        num_rows: 7811
    })
    test: Dataset({
        features: ['output', 'instruction', 'input'],
        num_rows: 1953
    })
})

### **Model & Tokenizer Implementation**

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [13]:
tokenizer = AutoTokenizer.from_pretrained("facebook/opt-350m", padding_side = 'left')

In [14]:
tokenizer

GPT2Tokenizer(name_or_path='facebook/opt-350m', vocab_size=50265, model_max_length=1000000000000000019884624838656, padding_side='left', truncation_side='right', special_tokens={'bos_token': '</s>', 'eos_token': '</s>', 'unk_token': '</s>', 'pad_token': '<pad>'}, added_tokens_decoder={
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
})

In [15]:
tokenizer.eos_token

'</s>'

In [16]:
model = AutoModelForCausalLM.from_pretrained("facebook/opt-350m")
model.to(device)

Loading weights: 100%|██████████| 388/388 [00:00<00:00, 21555.69it/s]


OPTForCausalLM(
  (model): OPTModel(
    (decoder): OPTDecoder(
      (embed_tokens): Embedding(50272, 512, padding_idx=1)
      (embed_positions): OPTLearnedPositionalEmbedding(2050, 1024)
      (project_out): Linear(in_features=1024, out_features=512, bias=False)
      (project_in): Linear(in_features=512, out_features=1024, bias=False)
      (layers): ModuleList(
        (0-23): 24 x OPTDecoderLayer(
          (self_attn): OPTAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (activation_fn): ReLU()
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=409

### Data Pre-Processing 

##### Formatting the prompt

In [17]:
def fromatting_prompt_with_response (mydataset):
    response = []
    
    for i in range(len(mydataset)):
        text = (
            f"### Instruction: \n{mydataset['instruction'][i]}"
            f"\n\n ### Response: \n{mydataset['output'][i]}</s>"
        )
        
        response.append(text)
    return response

In [23]:
def formatting_prompt_without_response(mydataset):
    
    response = []
    
    for i in range(len(mydataset)):
        text = (
            f"### Instruction: \n{mydataset['instruction'][i]}"
            f"\n\n### Response:\n"
        )
        
        response.append(text)
    return response

In [24]:
tokenized_text = tokenizer("Hi I am vishwa", return_tensors="pt")
tokenized_text

{'input_ids': tensor([[    2, 30086,    38,   524,   748,  1173,  2739]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]])}

In [21]:
tokenized_text['input_ids'][0]

2

In [31]:
expected_outputs = []

instruction_with_response = fromatting_prompt_with_response(dataset_test)
instruction_without_response = formatting_prompt_without_response(dataset_test)

for i in tqdm(range(len(instruction_with_response))):
    tokenized_instruction_with_response = tokenizer(instruction_with_response[i], return_tensors="pt")
    tokenized_instruction_without_response = tokenizer(instruction_without_response[i], return_tensors='pt')
    
    expected_output = tokenizer.decode(tokenized_instruction_with_response['input_ids'][0][len(tokenized_instruction_without_response['input_ids'][0])-1:],
                                       special_tokens = True)    
    expected_outputs.append(expected_output)
    

100%|██████████| 1953/1953 [00:01<00:00, 1312.18it/s]


In [34]:
instruction_without_response[0]

'### Instruction: \nName the most important benefit of using a database system.\n\n### Response:\n'

In [32]:
instruction_with_response[0]

'### Instruction: \nName the most important benefit of using a database system.\n\n ### Response: \nThe most important benefit of using a database system is the ability to store and retrieve data quickly and easily. Database systems also provide support for data security, data integrity, and concurrently accessing and modifying data from multiple systems.</s>'

In [33]:
expected_outputs[0]

'\nThe most important benefit of using a database system is the ability to store and retrieve data quickly and easily. Database systems also provide support for data security, data integrity, and concurrently accessing and modifying data from multiple systems.</s>'

# Converting Instructions into a PyTorch Dataset

Instead of keeping the instructions as a regular Python list, it is beneficial to convert them into a PyTorch `Dataset`. PyTorch models and training pipelines are designed to work efficiently with datasets that follow the `torch.utils.data.Dataset` structure.

To achieve this, a custom class called `ListDataset` is created by inheriting from `Dataset`. This class converts the `instructions` list into a dataset object that can be used directly in PyTorch workflows.

The `ListDataset` class typically implements:

- `__len__()` – Returns the total number of samples in the dataset.
- `__getitem__()` – Retrieves a specific sample using its index.

After converting the list into a `Dataset`, the data can be easily:

- Batched
- Shuffled
- Loaded efficiently with `DataLoader`
- Integrated into training and inference pipelines

This approach improves scalability, organization, and compatibility with industry-standard deep learning workflows in PyTorch.

In [36]:
class TemplateDataset(Dataset):
    
    def __init__(self, original_list):
        super().__init__()
        self.list = original_list
        
    def __len__(self):
        return len(self.list)

    def __getitem__(self, index):
        return self.list[index]